# Vision — Deteksi Objek (YOLOv8n)
**Kapan dipakai:** soal meminta kotak (bounding box) + kelas objek di gambar.

**Langkah:** buat `data.yaml` → ukur mAP model pretrained di data val → (opsional) fine-tune →
prediksi → jumlah objek per gambar.

**Data demo:** coco8 (8 gambar COCO, format YOLO). Folder data harus berisi `images/train`,
`images/val`, `labels/train`, `labels/val`. Bobot: `pretrained/yolov8n/yolov8n.pt`
(COCO 80 kelas). Untuk fine-tune: hapus `#` di sel **Fine-tune**, lalu jalankan ulang evaluasi.

In [1]:
import glob
import os

import pandas as pd
import torch
from ultralytics import YOLO

## Konfigurasi

In [2]:
DATA_FOLDER = "../../data_latihan/vision/detection/coco8"   # GANTI saat lomba
WEIGHTS = "../../pretrained/yolov8n/yolov8n.pt"
IMAGE_PATTERN = "*.jpg"     # GANTI bila gambar .png

EPOCHS = 3
IMAGE_SIZE = 640
MIN_CONFIDENCE = 0.25
DEVICE = "cpu"
if torch.cuda.is_available():
    DEVICE = 0

## Baca data

In [3]:
train_images = sorted(glob.glob(f"{DATA_FOLDER}/images/train/{IMAGE_PATTERN}"))
val_images = sorted(glob.glob(f"{DATA_FOLDER}/images/val/{IMAGE_PATTERN}"))
print("gambar train:", len(train_images), "| gambar val:", len(val_images))

gambar train: 4 | gambar val: 4


## Lihat satu file label
Format YOLO: satu baris per objek = `kelas x_tengah y_tengah lebar tinggi` (skala 0–1).

In [4]:
first_name = os.path.splitext(os.path.basename(val_images[0]))[0]
label_file = f"{DATA_FOLDER}/labels/val/{first_name}.txt"
print(open(label_file).read()[:200])

25 0.475759 0.414523 0.951518 0.672422
0 0.671279 0.617945 0.645759 0.726859



## Model (YOLOv8n pretrained, offline)

In [5]:
model = YOLO(WEIGHTS)
print("jumlah kelas:", len(model.names), "| contoh:", model.names[0], model.names[15])

jumlah kelas: 80 | contoh: person cat


## File data.yaml untuk ultralytics
Nama kelas diambil dari model. Saat lomba dengan kelas sendiri, tulis daftar kelas soal.

In [6]:
dataset_path = os.path.abspath(DATA_FOLDER).replace("\\", "/")   # YOLO butuh path lengkap
yaml_text = f"path: {dataset_path}\ntrain: images/train\nval: images/val\nnames:\n"
for class_number in model.names:
    yaml_text = yaml_text + f"  {class_number}: {model.names[class_number]}\n"
with open("data.yaml", "w") as file:
    file.write(yaml_text)
print(yaml_text[:200])

path: Z:/GEMASTIK-DATABASE/data_latihan/vision/detection/coco8
train: images/train
val: images/val
names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: airplane
  5: bus
  6: train
  7: truck


## Evaluasi (mAP di data val)

In [7]:
metrics = model.val(data="data.yaml", imgsz=IMAGE_SIZE, batch=8, workers=0, device=DEVICE,
                    project="runs", name="val", exist_ok=True, plots=False, verbose=False)
print("mAP50:", round(metrics.box.map50, 4), "| mAP50-95:", round(metrics.box.map, 4))

Ultralytics 8.4.172  Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8123MiB)


YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs


val: Fast image access  (ping: 0.00.0 ms, read: 471.2243.1 MB/s, size: 54.0 KB)


val: Scanning Z:\GEMASTIK-DATABASE\data_latihan\vision\detection\coco8\labels\val.cache... 4 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4/4 798.9Kit/s 0.0s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 3.6s/it 3.6s

                   all          4         17       0.62      0.833      0.888      0.635


Speed: 0.7ms preprocess, 43.3ms inference, 0.0ms loss, 1.9ms postprocess per image


mAP50: 0.8875 | mAP50-95: 0.6346


## Fine-tune (opsional, hapus `#` untuk memakai)

In [8]:
# model.train(data="data.yaml", epochs=EPOCHS, imgsz=IMAGE_SIZE, batch=8, workers=0,
#             device=DEVICE, project="runs", name="train", exist_ok=True, plots=False)

## Prediksi + hasil (jumlah objek terdeteksi per gambar)

In [9]:
results = model.predict(val_images, imgsz=IMAGE_SIZE, conf=MIN_CONFIDENCE, verbose=False)

file_names = []
box_counts = []
for image_path, image_result in zip(val_images, results):
    file_names.append(os.path.basename(image_path))
    box_counts.append(len(image_result.boxes))

result = pd.DataFrame()
result["file"] = file_names
result["jumlah_deteksi"] = box_counts
result.to_csv("hasil.csv", index=False)
result.head()

,file,jumlah_deteksi
0,000000000036.jpg,2
1,000000000042.jpg,3
2,000000000049.jpg,8
3,000000000061.jpg,2
